# Chapter 7: Claim Verification and Calibrated Abstention

**The question.** A validation report is full of numbers. How can we check automatically that each number is right, and that it is attached to the right thing? This chapter builds a *verifier*. It looks a claimed number up in the evidence store from Chapter 6 and returns one of four verdicts:

- *Supported*: the store has a record for the claim, and the values agree.
- *Contradicted*: the store has a record, and the values differ.
- *Absent*: the store has no record, because the study never measured it. That does not make the claim false.
- *Unscorable*: the claim is malformed, for example a number written as text, so it cannot be compared at all.

**The steps.**

1. Run 17 test claims whose right verdicts are known.
2. Recheck the book's worked example on the mixture-model study.
3. Show the difference between a wrong claim and an unmeasured one.
4. Damage a record on purpose and check that the verifier refuses to give a verdict.
5. Show what a Supported verdict does not prove.

**Result.** The verifier gets all 17 test claims right, catches a correct number attached to the wrong measurement and stops when a record is damaged. Checking each number, however, cannot catch a wrong comparison between numbers.

**Map of this notebook.**

| Step | Question | Where to look |
|---|---|---|
| 1 | Does the verifier give the right verdict on known cases? | `expected` and `actual` columns |
| 2 | Does it reproduce the book's example? | the six printed lines |
| 3 | How is a wrong claim different from an unmeasured one? | the four printed verdicts |
| 4 | What happens when a record is damaged? | the two error messages |
| 5 | What does Supported not prove? | the seed-73102 contrasts |

**Terms.** A *claim* is an address plus a claimed value, for example `region:confirmation/reflection_73101/contrast` with value 0.058326. The *address* (`id`) names the study (`region:` for the regional search of Chapters 4 and 5, `moe:` for the mixture-model study of Appendix B), the data split, the object and the measurement. `reflection_73101` is the Chapter 5 union branch at seed 73101. Two numbers *agree* when they differ by at most 0.0000005, which allows rounding to six decimals and nothing more; counts must match exactly. The data are simulated.

The notebook reads saved evidence only; it fits no model and calls no language model. Run the cells in order; the `assert` lines check each verdict against its expected or saved value.

## Setup and source checks

Loads the evidence store, the verifier and the saved results, and checks that the files have not changed. Select the kernel of the environment where `aiv` is installed.

In [1]:
from pathlib import Path
import os
import platform
import sys
import importlib.util

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
HELPERS = ROOT / "book/notebooks/regional_case"
for path in (ROOT, ROOT / "book", HELPERS):
    value = str(path)
    if value in sys.path:
        sys.path.remove(value)
    sys.path.insert(0, value)

# Load current helper files even if this kernel cached an older notebook revision.
def load_helper_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    try:
        spec.loader.exec_module(module)
    except Exception:
        sys.modules.pop(name, None)
        raise
    return module

load_helper_module("companion", HELPERS / "companion.py")
load_helper_module("ch02_helpers", HELPERS / "ch02_helpers.py")
load_helper_module("weighted_leaf_geometry", ROOT / "book/weighted_leaf_geometry.py")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
from companion import RUN, read, load, frontier, confirmation, measure, assignments
pd.set_option("display.float_format", "{:.6f}".format)
protocol, results, frozen = load()
print("Evidence:", RUN.relative_to(ROOT).as_posix())
print("Mode: saved-evidence replay; no fitting or live LLM calls")


Python: 3.12.3
Data root found from: repository checkout
Evidence: book/evidence/aml_medoid_loop_leaf_output_20260921
Mode: saved-evidence replay; no fitting or live LLM calls


## Step 1: Test the verifier on 17 claims with known answers

`cases()` supplies 17 made-up claims, each with the verdict it should get: correct values from both studies, a wrong value, correct numbers at the wrong address, requests for results nobody measured and three malformed claims. The cell runs the verifier on each claim and checks that it returns the expected verdict. Section 7.2 of the book defines the four verdicts.

In [2]:
from book.parallel_evidence.build import OUT, cases, verify
store=read(OUT/"store.json")
rows=[]
for name,id_,value,expected in cases():
    verdict=verify(store,{"id":id_,"value":value})["verdict"]
    assert verdict==expected
    rows.append(dict(case=name,id=id_,value=value,expected=expected,actual=verdict))
suite=pd.DataFrame(rows)
display(suite)
counts=suite["actual"].value_counts().reindex(["Supported","Contradicted","Absent","Unscorable"])
assert counts.tolist()==[7,4,3,3]
display(counts.rename("cases"))

,case,id,value,expected,actual
0,moe_correct,moe:confirmation/llm_compromise/worst_logloss,0.103333,Supported,Supported
1,moe_false,moe:confirmation/llm_compromise/worst_logloss,0.400000,Contradicted,Contradicted
2,moe_missing,moe:experiment/fairness_passed,True,Absent,Absent
3,regional_loss,region:confirmation/reflection_73101/brier,0.072851,Supported,Supported
4,regional_contrast,region:confirmation/reflection_73101/contrast,0.058326,Supported,Supported
5,regional_share,region:confirmation/reflection_73101/share,0.115220,Supported,Supported
6,regional_count,region:protocol/confirmation_n,200000,Supported,Supported
7,admission,region:admission/admitted,True,Supported,Supported
8,paired_interval,region:comparison/reflection-continuation_7310...,"[-0.001723, 0.002495]",Supported,Supported
9,wrong_metric,region:confirmation/reflection_73101/brier,0.058326,Contradicted,Contradicted


actual
Supported       7
Contradicted    4
Absent          3
Unscorable      3
Name: cases, dtype: int64

**Read the output.** **In one line: the verifier gives the expected verdict on all 17 claims.**

**Columns.** `case`: a short name for the test. `id`: the address being checked; long addresses are cut off with `...`. `value`: the claimed value. `expected`: the verdict the claim should get. `actual`: the verdict it got. The second table counts the verdicts: 7 Supported, 4 Contradicted, 3 Absent and 3 Unscorable.

**Notice.** Rows 9 to 11 (`wrong_metric`, `wrong_split` and `wrong_study`) all claim 0.058326, the correct contrast of the seed-73101 union-branch region, but at the wrong address: a Brier loss, the discovery split and the other study. A number that is right somewhere is still Contradicted in the wrong place. Rows 14 to 16 are Unscorable: the text `"0.058326"`, `True` as a sample size and an interval with only one number. These tests have authored answers, so they check that the code follows the rules, not how well claims in free text would be verified.

## Step 2: Recheck the worked example from the book

The chapter's worked example checks six claims about the mixture-model study of Appendix B: four correct, one false and one about a fairness assessment the study never did. The saved verdicts are in `claim_verdicts.json`. The cell does not trust them: it looks each value up again in its source file and recomputes the verdict.

In [3]:
from book.parallel_evidence.build import resolve, equal
from companion import sha
saved=read(ROOT/"book/evidence/aml_current/claim_verdicts.json")
lines=[]
for row in saved:
    evidence=row["evidence"]
    if evidence is None:
        verdict="Absent"
    else:
        assert sha(ROOT/evidence["source"])==evidence["source_sha256"]
        value=resolve(read(ROOT/evidence["source"]),evidence["pointer"])
        assert value==evidence["value"]
        verdict="Supported" if equal(row["claimed"],value) else "Contradicted"
    assert verdict==row["verdict"]
    lines.append(f"{verdict}: {row['entity']} {row['field']} {row['claimed']}")
assert "\n".join(lines)==(ROOT/"book/evidence/aml_current/claim_output.txt").read_text(encoding="utf-8").strip()
print("\n".join(lines))

Supported: llm_compromise overall_logloss 0.09826
Supported: llm_compromise worst_logloss 0.103333
Supported: experiment confirmation_n 200000
Supported: full_compromise worst_logloss 0.103417
Contradicted: llm_compromise worst_logloss 0.4
Absent: experiment fairness_passed True


**Read the output.** **In one line: the recomputed verdicts match the six lines printed in the chapter.**

**Lines.** Each line gives the verdict, the object, the measurement and the claimed value. `llm_compromise` and `full_compromise` are two model versions compared in the mixture-model study, and `experiment` is the study itself.

**Notice.** 0.09826 is Supported because the record is 0.098260, the same to six decimals. The claim of 0.4 for the worst-group logloss is Contradicted; the record is 0.103333. The fairness claim is Absent: no fairness assessment was recorded, which says nothing about whether the model is fair or unfair.

## Step 3: Wrong versus missing

Contradicted and Absent call for different fixes. Contradicted means the report misstates a recorded number, so the number must be corrected. Absent means the report claims something that was never measured, so the claim must be deleted or new evidence collected. The cell checks the contrast 0.058326 at its own address and at the Brier-loss address of the same region, then asks for two results the study never measured: a gain from repairing the model and a confirmed cause behind the Chapter 5 proposal.

In [4]:
claim={"id":"region:confirmation/reflection_73101/contrast","value":0.058326}
print(verify(store,claim)["verdict"])
claim["id"]="region:confirmation/reflection_73101/brier"
print(verify(store,claim)["verdict"])
for id_,value in [("region:experiment/predictor_repair_gain",.01),("region:experiment/fragmentation_cause_confirmed",True)]:
    print(id_,verify(store,{"id":id_,"value":value})["verdict"])

Supported
Contradicted
region:experiment/predictor_repair_gain Absent
region:experiment/fragmentation_cause_confirmed Absent


**Read the output.** **In one line: the same number is Supported in the right place and Contradicted in the wrong place, and unmeasured results are Absent.**

**Lines.** First, 0.058326 as the contrast: Supported. Second, 0.058326 as the Brier loss: Contradicted, because the record is 0.072851. Third and fourth, the repair gain and the confirmed cause: Absent.

**Notice.** The searches of Chapters 4 and 5 measured where the model does badly. They never retrained the model and never tested why the errors happen, so a report claiming a repair gain or a cause would be unsupported. The verifier flags such a claim without calling it false.

## Step 4: Damage a record on purpose

A verdict means something only if the record behind it is intact. The cell copies the store and damages one record twice: once its fingerprint (the SHA-256 digest of its source file) and once its value, set to −1. A correct claim is then checked against each damaged copy. The saved store is not changed.

In [5]:
from copy import deepcopy
key="region:confirmation/reflection_73101/contrast"
for field,bad in [("source_sha256","damaged"),("value",-1.)]:
    broken=deepcopy(store);broken["facts"][key][field]=bad
    try:
        verify(broken,{"id":key,"value":.058326})
    except ValueError as e:
        print(type(e).__name__,str(e))
    else:
        raise AssertionError("Corrupt record accepted")

ValueError Source hash mismatch: book/evidence/aml_medoid_loop_leaf_output_20260921/results.json
ValueError Source pointer/value mismatch: region:confirmation/reflection_73101/contrast


**Read the output.** **In one line: the verifier refuses to give a verdict on a damaged record.**

**Lines.** The two lines are error messages. `Source hash mismatch`: the fingerprint no longer matches the source file. `Source pointer/value mismatch`: the stored value no longer matches the value in the source file, which still holds the original contrast.

**Notice.** Returning Contradicted or Absent here would blame the claim for a broken pipeline. Stopping with an error forces the pipeline to be repaired before any claim is judged.

## Step 5: What Supported does not prove

Supported means a number was copied correctly. It does not mean the sentence around the number is right. The cell checks two records about the Chapter 5 proposal: that the union rule was approved, and the range of plausible values for union branch minus continuation at seed 73101. It then prints both branches' contrasts at seed 73102.

In [6]:
for id_ in ["region:admission/admitted","region:comparison/reflection-continuation_73101/simultaneous_ci"]:
    value=store["facts"][id_]["value"]
    print(id_,value,verify(store,{"id":id_,"value":value})["verdict"])
display(pd.Series({arm:store["facts"][f"region:confirmation/{arm}_73102/contrast"]["value"] for arm in ["reflection","continuation"]},name="confirmation contrast, seed 73102"))

region:admission/admitted True Supported
region:comparison/reflection-continuation_73101/simultaneous_ci [-0.0017229699434947615, 0.002494938298177806] Supported


reflection     0.053993
continuation   0.057581
Name: confirmation contrast, seed 73102, dtype: float64

**Read the output.** **In one line: every number can be Supported while the sentence that uses them is wrong.**

**Lines and table.** Both records print as Supported. The table gives the contrast on new customers at seed 73102 for `reflection` (the union branch, 0.053993) and `continuation` (0.057581).

**Notice.** Approval only means the rule could run fairly, and the range [−0.001723, 0.002495] contains zero, so neither record shows that the union rule did better. Suppose a report says "the union branch had the larger contrast at every seed". Each number it cites could be Supported, yet at seed 73102 the union branch is lower, so the sentence is wrong. Checking numbers does not check comparisons.

## What we learned

The verifier tells apart a correct value, a value at the wrong address, a result that was never measured and a malformed claim, and it stops when a record is damaged. It got all 17 test claims right and reproduced the book's worked example.

Its limits: every claim here came with an exact address and a typed value. Turning a phrase such as "the weak region's loss" into the right address, and checking comparisons between numbers, are separate problems these checks do not cover. Chapter 8 tests the verifier against deliberately tricky inputs, and Chapter 10 uses it on generated report text.


## What to try

1. **Counts must match exactly.** Predict the verdict for `{"id": "region:confirmation/reflection_73101/n", "value": 23044.0}`, then run it. How does the rule for counts explain the result?
2. **Tolerance.** What is the smallest change to 0.058326 that turns the contrast claim from Supported to Contradicted? Why does the tolerance allow for rounding but not for sampling noise?
3. **Wrong and missing.** Write one false claim and one unmeasured claim about the seed-73103 union-branch region. Verify both. What should the report's author do in each case?
4. **Scope.** Why does a Supported verdict for every number in a paragraph not make the paragraph right?
